#Preparando ambiente

In [5]:
!pip install transformers==5.0.0rc0
!pip install mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.4/533.4 kB 52.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 152.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 149.9 MB/s eta 0:00:00


In [1]:
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend, FineGrainedFP8Config
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import random
import torch
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Carregando dataset ASSIN2

In [3]:
def gera_df():

  splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet'}
  df_assin_2_treino = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["train"])
  df_assin_2_teste = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["test"])
  df_assin_2_val = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["validation"])

  df_assin_2 = pd.concat([df_assin_2_treino, df_assin_2_teste, df_assin_2_val])

  df_assin_2['premise_hypothesis'] = df_assin_2['premise'] + ' ' + df_assin_2['hypothesis']

  occurrence_counts = df_assin_2['premise_hypothesis'].value_counts().reset_index()
  occurrence_counts.columns = ['premise_hypothesis', 'occurrence_count']

  df_assin_2 = df_assin_2.merge(occurrence_counts, on='premise_hypothesis', how='left')

  df_assin_2 = df_assin_2.loc[df_assin_2['occurrence_count'] == 1]

  df_assin_2 = df_assin_2[['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score',
        'entailment_judgment']]

  df_assin_2.reset_index(drop=True, inplace=True)

  return df_assin_2

In [4]:
def extract_response_character(response_text):
  """
  Extrai o caractere '0' ou '1' da resposta do modelo.
  Args:
    response_text (str): A string de resposta do modelo.
  Returns:
    str: '0' ou '1' se encontrado, caso contrário, None.
  """
  # Use regex to find '0' or '1' potentially preceded by whitespace at the beginning of the string
  match = re.search(r'^[\s]*([01])', response_text)
  if match:
    return match.group(1)
  return None

In [5]:
def zero_shot_prompt(premise, hypothesis):
  return f"""
   Você é um sistema de Reconhecimento de Inferência Textual (RTE) em Português Brasileiro.

    Tarefa:
    Dada uma PREMISSA e uma HIPÓTESE, responda *apenas* com um único caractere:
    - 0 se a hipótese não é inferida da premissa.
    - 1 se a hipótese é logicamente inferida da premissa.

    Regras obrigatórias:
    - NÃO explique.
    - NÃO acrescente texto.
    - NÃO repita o enunciado.
    - NÃO responda nada além de 0 ou 1.

    Premissa: {premise}
    Hipótese: {hypothesis}

    Resposta:
    """

#Ministral 3 - 3B - Reasoning

In [6]:
model_id = "mistralai/Ministral-3-3B-Reasoning-2512"
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

In [7]:
df_assin_2 = gera_df()

num_records = len(df_assin_2)
random_index = random.randint(0, num_records - 1)
random_index = 2

premissa = df_assin_2.iloc[random_index]['premise']
hipotese = df_assin_2.iloc[random_index]['hypothesis']
prompt = zero_shot_prompt(premissa, hipotese)


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=10)
prompt_len = inputs["input_ids"].shape[1]
generated_tokens = output[0][prompt_len:]
resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f'Prompt: {prompt}')
print(f'Resposta: {resp}')

Prompt: 
   Você é um sistema de Reconhecimento de Inferência Textual (RTE) em Português Brasileiro.

    Tarefa:
    Dada uma PREMISSA e uma HIPÓTESE, responda *apenas* com um único caractere:
    - 0 se a hipótese não é inferida da premissa.
    - 1 se a hipótese é logicamente inferida da premissa.

    Regras obrigatórias:
    - NÃO explique.
    - NÃO acrescente texto.
    - NÃO repita o enunciado.
    - NÃO responda nada além de 0 ou 1.

    Premissa: Uma pessoa tem cabelo loiro e esvoaçante e está tocando violão
    Hipótese: Um guitarrista tem cabelo loiro e esvoaçante

    Resposta:
    
Resposta:  1


##Teste de Consistência

In [8]:
df_assin_2_first_500 = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_first_500)):
    premissa = df_assin_2_first_500.iloc[i]['premise']
    hipotese = df_assin_2_first_500.iloc[i]['hypothesis']

    prompt = zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=10)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'mistral3b_reasoning_{j}'

    df_assin_2_first_500.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2_first_500.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_mistral3b_reasoning_consistencia.csv')

/tmp/ipython-input-4030351167.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 02:22:22
100 - 2026-01-11 02:22:51
200 - 2026-01-11 02:23:22
300 - 2026-01-11 02:23:52
400 - 2026-01-11 02:24:23


/tmp/ipython-input-4030351167.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 02:24:54
100 - 2026-01-11 02:25:25
200 - 2026-01-11 02:25:56
300 - 2026-01-11 02:26:27
400 - 2026-01-11 02:26:58


/tmp/ipython-input-4030351167.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 02:27:29
100 - 2026-01-11 02:28:00
200 - 2026-01-11 02:28:31
300 - 2026-01-11 02:29:02
400 - 2026-01-11 02:29:33


/tmp/ipython-input-4030351167.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 02:30:04
100 - 2026-01-11 02:30:35
200 - 2026-01-11 02:31:06
300 - 2026-01-11 02:31:37
400 - 2026-01-11 02:32:08


/tmp/ipython-input-4030351167.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_first_500.loc[i, column_name] = resp


0 - 2026-01-11 02:32:39
100 - 2026-01-11 02:33:10
200 - 2026-01-11 02:33:41
300 - 2026-01-11 02:34:12
400 - 2026-01-11 02:34:43


#Loop de aplicação do prompt em todo o dataset

In [8]:
for i in range(50):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  df_assin_2.loc[i, 'ministral_3b_reasoning'] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

0 - 2026-01-11 15:23:23


In [10]:
df_assin_2.head(20)

,sentence_pair_id,premise,hypothesis,relatedness_score,entailment_judgment,ministral_3b_reasoning
0,1,Uma criança risonha está segurando uma pistola...,Uma criança está segurando uma pistola de água,4.5,1,1
1,2,Os homens estão cuidadosamente colocando as ma...,Os homens estão colocando bagagens dentro do p...,4.5,1,1
2,3,Uma pessoa tem cabelo loiro e esvoaçante e est...,Um guitarrista tem cabelo loiro e esvoaçante,4.7,1,1
3,4,Batatas estão sendo fatiadas por um homem,O homem está fatiando a batata,4.7,1,1
4,5,Um caminhão está descendo rapidamente um morro,Um caminhão está rapidamente descendo o morro,4.9,1,1
5,6,Um surfista está pegando uma grande onda atrav...,Uma onda grande está sendo surfada por um surf...,4.8,1,1
6,7,Três cachorros pequeninos estão cheirando algo,Três cachorros estão cheirando alguma coisa,4.6,1,1
7,8,Alguns adultos estão sentados nas cadeiras e e...,Dois adultos estão sentados em cadeiras e estã...,4.7,1,0
8,9,A gruta com interior rosa está sendo escalada ...,"Quatro crianças do Oriente Médio, três meninas...",4.9,1,1
9,10,O homem com um chapéu duro está dançando,Um homem com um chapéu duro está dançando,4.9,1,1


In [ ]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  df_assin_2.loc[i, 'ministral_3b_reasoning'] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_mistral_3b_reasoning.csv')

0 - 2025-12-23 00:13:12
100 - 2025-12-23 00:13:28
200 - 2025-12-23 00:13:44
300 - 2025-12-23 00:14:00
400 - 2025-12-23 00:14:16
500 - 2025-12-23 00:14:32
600 - 2025-12-23 00:14:47
700 - 2025-12-23 00:15:03
800 - 2025-12-23 00:15:19
900 - 2025-12-23 00:15:34
1000 - 2025-12-23 00:15:50
1100 - 2025-12-23 00:16:06
1200 - 2025-12-23 00:16:21
1300 - 2025-12-23 00:16:37
1400 - 2025-12-23 00:16:53
1500 - 2025-12-23 00:17:09
1600 - 2025-12-23 00:17:25
1700 - 2025-12-23 00:17:40
1800 - 2025-12-23 00:17:56
1900 - 2025-12-23 00:18:12
2000 - 2025-12-23 00:18:28
2100 - 2025-12-23 00:18:44
2200 - 2025-12-23 00:19:00
2300 - 2025-12-23 00:19:15
2400 - 2025-12-23 00:19:31
2500 - 2025-12-23 00:19:47
2600 - 2025-12-23 00:20:03
2700 - 2025-12-23 00:20:19
2800 - 2025-12-23 00:20:35
2900 - 2025-12-23 00:20:51
3000 - 2025-12-23 00:21:07
3100 - 2025-12-23 00:21:23
3200 - 2025-12-23 00:21:38
3300 - 2025-12-23 00:21:54
3400 - 2025-12-23 00:22:10
3500 - 2025-12-23 00:22:25
3600 - 2025-12-23 00:22:41
3700 - 2025-1

In [ ]:
df_assin_2.head()

,sentence_pair_id,premise,hypothesis,relatedness_score,entailment_judgment,ministral_3b_reasoning
0,1.0,Uma criança risonha está segurando uma pistola...,Uma criança está segurando uma pistola de água,4.5,1.0,1
1,2.0,Os homens estão cuidadosamente colocando as ma...,Os homens estão colocando bagagens dentro do p...,4.5,1.0,1
2,3.0,Uma pessoa tem cabelo loiro e esvoaçante e est...,Um guitarrista tem cabelo loiro e esvoaçante,4.7,1.0,1
3,4.0,Batatas estão sendo fatiadas por um homem,O homem está fatiando a batata,4.7,1.0,1
4,5.0,Um caminhão está descendo rapidamente um morro,Um caminhão está rapidamente descendo o morro,4.9,1.0,1


# Carregando Ministral 3 - 3B - Base

In [ ]:
model_id = "mistralai/Ministral-3-3B-Base-2512"
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

In [ ]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  df_assin_2.loc[i, 'ministral_3b_instruct'] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_mistral_3b_base.csv')

0 - 2025-12-23 00:38:20
100 - 2025-12-23 00:39:00
200 - 2025-12-23 00:39:40
300 - 2025-12-23 00:40:20
400 - 2025-12-23 00:41:00
500 - 2025-12-23 00:41:41
600 - 2025-12-23 00:42:21
700 - 2025-12-23 00:43:01
800 - 2025-12-23 00:43:41
900 - 2025-12-23 00:44:21
1000 - 2025-12-23 00:45:01
1100 - 2025-12-23 00:45:42
1200 - 2025-12-23 00:46:22
1300 - 2025-12-23 00:47:02
1400 - 2025-12-23 00:47:42
1500 - 2025-12-23 00:48:22
1600 - 2025-12-23 00:49:02
1700 - 2025-12-23 00:49:42
1800 - 2025-12-23 00:50:23
1900 - 2025-12-23 00:51:03
2000 - 2025-12-23 00:51:42
2100 - 2025-12-23 00:52:22
2200 - 2025-12-23 00:53:02
2300 - 2025-12-23 00:53:42
2400 - 2025-12-23 00:54:22
2500 - 2025-12-23 00:55:02
2600 - 2025-12-23 00:55:42
2700 - 2025-12-23 00:56:22
2800 - 2025-12-23 00:57:03
2900 - 2025-12-23 00:57:43
3000 - 2025-12-23 00:58:23
3100 - 2025-12-23 00:59:04
3200 - 2025-12-23 00:59:44
3300 - 2025-12-23 01:00:24
3400 - 2025-12-23 01:01:04
3500 - 2025-12-23 01:01:44
3600 - 2025-12-23 01:02:24
3700 - 2025-1

# Carregando Ministral 3 - 8B - Base

In [ ]:
model_id = "mistralai/Ministral-3-8B-Base-2512"
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

In [ ]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  df_assin_2.loc[i, 'ministral_8b_instruct'] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_mistral_8b_base.csv')

0 - 2025-12-23 02:02:50
100 - 2025-12-23 02:03:53
200 - 2025-12-23 02:04:55
300 - 2025-12-23 02:05:58
400 - 2025-12-23 02:07:00
500 - 2025-12-23 02:08:03
600 - 2025-12-23 02:09:05
700 - 2025-12-23 02:10:07
800 - 2025-12-23 02:11:10
900 - 2025-12-23 02:12:12
1000 - 2025-12-23 02:13:14
1100 - 2025-12-23 02:14:16
1200 - 2025-12-23 02:15:19
1300 - 2025-12-23 02:16:21
1400 - 2025-12-23 02:17:23
1500 - 2025-12-23 02:18:26
1600 - 2025-12-23 02:19:28
1700 - 2025-12-23 02:20:31
1800 - 2025-12-23 02:21:33
1900 - 2025-12-23 02:22:36
2000 - 2025-12-23 02:23:38
2100 - 2025-12-23 02:24:40
2200 - 2025-12-23 02:25:43
2300 - 2025-12-23 02:26:45
2400 - 2025-12-23 02:27:48
2500 - 2025-12-23 02:28:50
2600 - 2025-12-23 02:29:53
2700 - 2025-12-23 02:30:55
2800 - 2025-12-23 02:31:58
2900 - 2025-12-23 02:33:00
3000 - 2025-12-23 02:34:02
3100 - 2025-12-23 02:35:05
3200 - 2025-12-23 02:36:07
3300 - 2025-12-23 02:37:10
3400 - 2025-12-23 02:38:12
3500 - 2025-12-23 02:39:14
3600 - 2025-12-23 02:40:16
3700 - 2025-1

# Carregando Ministral 3 - 8B - Reasoning

In [ ]:
model_id = "mistralai/Ministral-3-8B-Reasoning-2512"
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

In [ ]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  df_assin_2.loc[i, 'ministral_8b_reasoning'] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_mistral_8b_reasoning.csv')

0 - 2025-12-23 04:42:25
100 - 2025-12-23 04:42:55
200 - 2025-12-23 04:43:26
300 - 2025-12-23 04:43:56
400 - 2025-12-23 04:44:27
500 - 2025-12-23 04:44:57
600 - 2025-12-23 04:45:28
700 - 2025-12-23 04:45:58
800 - 2025-12-23 04:46:28
900 - 2025-12-23 04:46:58
1000 - 2025-12-23 04:47:28
1100 - 2025-12-23 04:47:58
1200 - 2025-12-23 04:48:29
1300 - 2025-12-23 04:48:59
1400 - 2025-12-23 04:49:29
1500 - 2025-12-23 04:50:00
1600 - 2025-12-23 04:50:30
1700 - 2025-12-23 04:51:00
1800 - 2025-12-23 04:51:31
1900 - 2025-12-23 04:52:01
2000 - 2025-12-23 04:52:32
2100 - 2025-12-23 04:53:02
2200 - 2025-12-23 04:53:32
2300 - 2025-12-23 04:54:03
2400 - 2025-12-23 04:54:33
2500 - 2025-12-23 04:55:04
2600 - 2025-12-23 04:55:34
2700 - 2025-12-23 04:56:04
2800 - 2025-12-23 04:56:35
2900 - 2025-12-23 04:57:06
3000 - 2025-12-23 04:57:36
3100 - 2025-12-23 04:58:07
3200 - 2025-12-23 04:58:37
3300 - 2025-12-23 04:59:08
3400 - 2025-12-23 04:59:38
3500 - 2025-12-23 05:00:08
3600 - 2025-12-23 05:00:37
3700 - 2025-1